# Original-architecture fusion — reviewer-aligned frozen-branch revision

This notebook preserves the fusion architecture used for the submitted manuscript while consuming the **already-finalized acoustic and linguistic branches**. Neither branch is retrained.

## Preserved submitted fusion design
- Four meta-features: acoustic probability, linguistic probability, absolute difference, and product.
- Fusion MLP: 4 → 16 → 8, BatchNorm/ReLU/Dropout(0.3).
- Two outputs: binary classification and PHQ-8 severity regression.
- Joint objective: class-weighted BCE + 0.3 × MSE on sigmoid-scaled PHQ-8/24.
- AdamW, learning rate 1e-3, weight decay 1e-2, cosine schedule, 200 epochs, five-seed ensemble.

## Reviewer-critical evaluation boundaries
- Branches are frozen; their saved participant-level OOF probabilities are used for development and saved test probabilities for held-out evaluation.
- Fusion development assessment uses leave-one-participant-out refitting of the fusion head over fixed branch OOF inputs. This is **not fully nested LOO of the complete multimodal pipeline**, because branch models are not retrained within each fusion fold. This limitation is explicitly recorded.
- Thresholds are selected from development-only fusion OOF probabilities. Test labels are not used for threshold selection or calibration.
- Test labels are loaded only after test probabilities and thresholds have been frozen.
- No training AUC is compared with held-out test AUC. The independent held-out test is the primary generalisation estimate.

**Do not claim full nested LOO or clinical diagnostic performance.** Describe the contribution as an evaluated multimodal implementation/stacking framework, distinguishing it from established acoustic and language encoders.


## PHQ-8 severity method
The severity estimate in this notebook is generated by the fusion model's multi-task regression head (sigmoid output scaled to 0–24). Isotonic regression is **not** used as a substitute severity head in this implementation. The manuscript must use this same description and report MAE/RMSE/R² from the held-out cohort.


## Verified input compatibility

The accompanying input set currently has 107 matched development participants and 46 matched held-out participants with both branch probabilities. The held-out label file contains 47 IDs; participant 334 has no matching pair of branch probabilities and is excluded from paired branch/fusion test metrics. The notebook prints this explicitly and records unmatched IDs. Do not describe the test cohort as 47 evaluable participants.


**Execution note:** run cells from top to bottom. The notebook checks required files and column schemas before training.

In [1]:
import os, glob, json, math, time, random, pickle, hashlib, platform, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, accuracy_score, f1_score, brier_score_loss)
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

SEED = 42
N_SEEDS = 5
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-2
DROPOUT = 0.3
HIDDEN = (16, 8)
ALPHA = 0.3
THRESHOLD_FIXED= 0.5
N_BOOT = 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.makedirs(WORKING, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)

Device: cpu | PyTorch: 2.11.0+cpu


## 1. Load the frozen branch artifacts

Required files:
- `acoustic_paper_strict_loo_oof.csv`
- `linguistic_oof_predictions_v11_final.csv`
- `acoustic_paper_final_test_predictions.csv`
- `linguistic_test_predictions_v11_final.csv`
- `full_test_split.csv`

The linguistic test-prediction CSV was exported from the supplied v11 model bundle. No branch is retrained in this notebook.
**Input compatibility:** `linguistic_test_predictions_v11_final.csv` is optional when the v11 pickle bundle contains `test_results`. The notebook uses the participant-ID intersection for test evaluation and prints IDs omitted from the matched evaluation. The DAIC-WOZ held-out label file may contain 47 rows while branch predictions contain 46; this is reported rather than treated as a software crash.


In [2]:
def locate(name):
    roots = ["/kaggle/input", "/kaggle/working", "/mnt/data", WORKING, os.getcwd()]
    hits = []
    for root in roots:
        if os.path.isdir(root):
            hits += glob.glob(os.path.join(root, "**", name), recursive=True)
    hits = sorted(set(hits), key=os.path.getmtime, reverse=True)
    if not hits:
        raise FileNotFoundError(f"Upload or attach {name!r} as a notebook input.")
    return hits[0]

p_aoof = locate("acoustic_paper_strict_loo_oof.csv")
p_loof = locate("linguistic_oof_predictions_v11_final.csv")
p_atest = locate("acoustic_paper_final_test_predictions.csv")
# The final v11 bundle can supply test probabilities, so the CSV is optional.
try:
    p_ltest = locate("linguistic_test_predictions_v11_final.csv")
except FileNotFoundError:
    p_ltest = None
p_labels = locate("full_test_split.csv")

def pid_int(x): return int(float(str(x).strip()))

aoof = pd.read_csv(p_aoof); loof = pd.read_csv(p_loof)
at = pd.read_csv(p_atest)
if p_ltest is not None and os.path.exists(p_ltest):
    lt = pd.read_csv(p_ltest)
else:
    # The final v11 bundle stores held-out test probabilities; use them without retraining ClinicalBERT.
    p_lbundle = locate("depression_model_v11_final_leakage_safe.pkl")
    with open(p_lbundle, "rb") as f:
        lb = pickle.load(f)
    tr = lb.get("test_results") if isinstance(lb, dict) else None
    if tr is None:
        raise FileNotFoundError(
            "Missing linguistic_test_predictions_v11_final.csv and the linguistic v11 bundle "
            "does not contain test_results. Export held-out probabilities from the already-trained "
            "final linguistic model; do not retrain the branch for this fusion notebook."
        )
    lt = pd.DataFrame({
        "Participant_ID": [pid_int(x) for x in tr["ids"]],
        "linguistic_test_probability": np.asarray(tr["prob"], dtype=float)
    })

# Normalize expected columns and fail with actionable messages rather than KeyError.
required = {
    "acoustic OOF": (aoof, ["Participant_ID", "PHQ8_Binary", "PHQ8_Score", "acoustic_oof_probability"]),
    "linguistic OOF": (loof, ["Participant_ID", "PHQ8_Binary", "PHQ8_Score", "linguistic_oof_probability"]),
    "acoustic test": (at, ["Participant_ID", "acoustic_test_probability"]),
    "linguistic test": (lt, ["Participant_ID", "linguistic_test_probability"]),
}
for label, (frame, cols) in required.items():
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise ValueError(f"{label} file is missing columns {missing}. Found: {list(frame.columns)}")
for d in [aoof, loof, at, lt]:
    d.columns = [str(c).strip() for c in d.columns]
    d["Participant_ID"] = d["Participant_ID"].map(pid_int)

dev = aoof.merge(loof, on=["Participant_ID", "PHQ8_Binary", "PHQ8_Score"],
                 how="inner", validate="one_to_one")
assert dev["Participant_ID"].is_unique
assert len(dev) >= 10, f"Matched development cohort unexpectedly small: {len(dev)}"
assert dev["PHQ8_Binary"].nunique() == 2
assert dev[["acoustic_oof_probability", "linguistic_oof_probability"]].notna().all().all()
assert dev["acoustic_oof_probability"].between(0,1).all()
assert dev["linguistic_oof_probability"].between(0,1).all()

test_branch = at[["Participant_ID", "acoustic_test_probability"]].merge(
    lt[["Participant_ID", "linguistic_test_probability"]],
    on="Participant_ID", how="inner", validate="one_to_one"
).sort_values("Participant_ID").reset_index(drop=True)
assert test_branch["Participant_ID"].is_unique
if test_branch.empty:
    raise ValueError("No overlapping held-out participants between acoustic and linguistic predictions.")
assert test_branch[["acoustic_test_probability","linguistic_test_probability"]].notna().all().all()
assert test_branch[["acoustic_test_probability","linguistic_test_probability"]].apply(
    lambda c: c.between(0,1)).all().all()

print("Development participants with both OOF branch predictions:", len(dev))
print("Matched held-out test branch predictions:", len(test_branch))
print("Acoustic-only test IDs:", sorted(set(at.Participant_ID)-set(test_branch.Participant_ID)))
print("Linguistic-only test IDs:", sorted(set(lt.Participant_ID)-set(test_branch.Participant_ID)))
print("Development class counts:", dev["PHQ8_Binary"].value_counts().to_dict())

Development participants with both OOF branch predictions: 107
Matched held-out test branch predictions: 46
Acoustic-only test IDs: []
Linguistic-only test IDs: []
Development class counts: {0: 77, 1: 30}


## 2. Original fusion architecture and training objective

This cell retains the current submitted fusion-head design. No feature or architecture search is performed against the test set.

In [3]:
def make_meta_features(pa, pl, kind="full"):
    pa = np.asarray(pa, dtype=np.float32)
    pl = np.asarray(pl, dtype=np.float32)
    if kind == "full":
        return np.stack([pa, pl, np.abs(pa-pl), pa*pl], axis=1)
    if kind == "two":
        return np.stack([pa, pl], axis=1)
    raise ValueError(kind)

class FusionMLP(nn.Module):
    def __init__(self, d_in=4):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 8), nn.BatchNorm1d(8), nn.ReLU(), nn.Dropout(0.3)
        )
        self.cls = nn.Linear(8, 1)
        self.reg = nn.Linear(8, 1)
    def forward(self, x):
        z = self.body(x)
        return self.cls(z).squeeze(1), self.reg(z).squeeze(1)

def train_fusion(Xtr, ytr, scoretr, alpha=ALPHA, seed=SEED):
    torch.manual_seed(seed)
    X = torch.tensor(np.asarray(Xtr), dtype=torch.float32, device=DEVICE)
    y = torch.tensor(np.asarray(ytr), dtype=torch.float32, device=DEVICE)
    s = torch.tensor(np.asarray(scoretr, dtype=np.float32)/24.0,
                     dtype=torch.float32, device=DEVICE)
    model = FusionMLP(X.shape[1]).to(DEVICE)
    pos = max(int(np.sum(ytr)), 1)
    neg = max(int(len(ytr)-np.sum(ytr)), 1)
    bce = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(neg/pos, dtype=torch.float32, device=DEVICE)
    )
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    for _ in range(EPOCHS):
        model.train()
        opt.zero_grad(set_to_none=True)
        cls, sev = model(X)
        loss = bce(cls, y) + alpha*nn.functional.mse_loss(torch.sigmoid(sev), s)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict_fusion(model, X):
    model.eval()
    x = torch.tensor(np.asarray(X), dtype=torch.float32, device=DEVICE)
    cls, sev = model(x)
    return torch.sigmoid(cls).cpu().numpy(), (24*torch.sigmoid(sev)).cpu().numpy()

def fit_ensemble(X, y, scores, X_test, alpha=ALPHA, kind="full"):
    models, pp, ss = [], [], []
    for k in range(N_SEEDS):
        m = train_fusion(X, y, scores, alpha=alpha, seed=SEED+k)
        p, s = predict_fusion(m, X_test)
        models.append(m); pp.append(p); ss.append(s)
    return np.mean(pp, axis=0), np.mean(ss, axis=0), models

## 3. Fit fusion on development OOF branch probabilities; generate label-free test predictions

The final branch models supplied the test probabilities. The test labels are not read in this section.

In [4]:
pa_dev = dev["acoustic_oof_probability"].to_numpy(float)
pl_dev = dev["linguistic_oof_probability"].to_numpy(float)
y_dev = dev["PHQ8_Binary"].to_numpy(int)
s_dev = dev["PHQ8_Score"].to_numpy(float)

pa_test = test_branch["acoustic_test_probability"].to_numpy(float)
pl_test = test_branch["linguistic_test_probability"].to_numpy(float)
X_dev = make_meta_features(pa_dev, pl_dev, "full")
X_test = make_meta_features(pa_test, pl_test, "full")

fusion_prob, fusion_severity, fusion_models = fit_ensemble(
    X_dev, y_dev, s_dev, X_test, alpha=ALPHA
)
test_predictions = pd.DataFrame({
    "Participant_ID": test_branch["Participant_ID"],
    "acoustic_probability": pa_test,
    "linguistic_probability": pl_test,
    "fusion_probability": fusion_prob,
    "fusion_severity_PHQ8": fusion_severity
})
pred_path = os.path.join(WORKING, "fusion_frozen_test_predictions_label_free.csv")
test_predictions.to_csv(pred_path, index=False)
torch.save([m.state_dict() for m in fusion_models],
           os.path.join(WORKING, "fusion_frozen_final_models.pt"))
print("Saved label-free predictions:", pred_path)
print("Test predictions generated:", len(test_predictions))

Saved label-free predictions: /kaggle/working/fusion_frozen_test_predictions_label_free.csv
Test predictions generated: 46


In [5]:
# Development-only fusion LOO predictions for threshold selection.
# Branch inputs remain the already-generated OOF probabilities; this is fusion-level cross-fitting,
# not a fully nested re-fit of the acoustic and linguistic branches.
from sklearn.metrics import roc_curve

loo_prob = np.zeros(len(dev), dtype=float)
loo_sev = np.zeros(len(dev), dtype=float)
for held in range(len(dev)):
    keep = np.ones(len(dev), dtype=bool)
    keep[held] = False
    m = train_fusion(X_dev[keep], y_dev[keep], s_dev[keep],
                     alpha=ALPHA, seed=SEED + held)
    pp, ss = predict_fusion(m, X_dev[held:held+1])
    loo_prob[held] = float(pp[0])
    loo_sev[held] = float(ss[0])

fpr_dev, tpr_dev, thresholds_dev = roc_curve(y_dev, loo_prob)
youden_idx = int(np.nanargmax(tpr_dev - fpr_dev))
THRESHOLD_YOUDEN_DEV = float(thresholds_dev[youden_idx])
THRESHOLD_FIXED = 0.5
print("Development-only fixed threshold:", THRESHOLD_FIXED)
print("Development-LOO Youden threshold:", THRESHOLD_YOUDEN_DEV)
# Threshold is now frozen using development-only out-of-fold fusion probabilities.
# Development metrics are printed after the metric helper is defined in the test-evaluation cell.


Development-only fixed threshold: 0.5
Development-LOO Youden threshold: 0.5800877809524536


## 4. Final held-out test evaluation

The independent test labels are loaded only now. All classification metrics below use the **predefined fixed threshold 0.5** for comparability. No threshold is optimized on test labels.

In [6]:
lab = pd.read_csv(p_labels)
lab.columns = [str(c).strip() for c in lab.columns]
lab["Participant_ID"] = lab["Participant_ID"].map(pid_int)
# DAIC-WOZ official held-out file uses PHQ_Binary / PHQ_Score; tolerate PHQ8_* aliases.
if "PHQ_Binary" not in lab.columns and "PHQ8_Binary" in lab.columns:
    lab = lab.rename(columns={"PHQ8_Binary":"PHQ_Binary", "PHQ8_Score":"PHQ_Score"})
need_lab = ["Participant_ID", "PHQ_Binary", "PHQ_Score"]
missing_lab = [c for c in need_lab if c not in lab.columns]
if missing_lab:
    raise ValueError(f"full_test_split.csv missing {missing_lab}; found columns {list(lab.columns)}")
test_eval = test_predictions.merge(
    lab[need_lab],
    on="Participant_ID", how="inner", validate="one_to_one"
)
if test_eval.empty:
    raise ValueError("No participant IDs overlap between test predictions and full_test_split.csv.")
if len(test_eval) < len(test_predictions):
    print("WARNING: some test predictions have no matching label:", sorted(set(test_predictions.Participant_ID)-set(test_eval.Participant_ID)))
y_test = test_eval["PHQ_Binary"].to_numpy(int)
score_test = test_eval["PHQ_Score"].to_numpy(float)
print("Matched labelled test participants:", len(test_eval))
print("Unmatched test participants:", sorted(set(lab.Participant_ID)-set(test_eval.Participant_ID)))
print("Test class counts:", pd.Series(y_test).value_counts().to_dict())

def wilson(k, n, z=1.959963984540054):
    if n == 0: return (np.nan, np.nan)
    ph = k/n; d = 1+z*z/n
    c = (ph+z*z/(2*n))/d
    h = z*np.sqrt((ph*(1-ph)+z*z/(4*n))/n)/d
    return c-h, c+h

def classification_metrics(y, p, threshold=THRESHOLD_FIXED):
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    div = lambda a,b: a/b if b else np.nan
    return {
        "threshold": threshold, "AUC": roc_auc_score(y,p),
        "PR_AUC": average_precision_score(y,p),
        "accuracy": accuracy_score(y,pred),
        "sensitivity": div(tp,tp+fn), "specificity": div(tn,tn+fp),
        "PPV": div(tp,tp+fp), "NPV": div(tn,tn+fn),
        "F1": f1_score(y,pred,zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "sensitivity_95CI": wilson(tp,tp+fn),
        "specificity_95CI": wilson(tn,tn+fp),
        "PPV_95CI": wilson(tp,tp+fp), "NPV_95CI": wilson(tn,tn+fn),
        "Brier": brier_score_loss(y,p)
    }

model_probs = {
    "Acoustic": test_eval["acoustic_probability"].to_numpy(float),
    "Linguistic": test_eval["linguistic_probability"].to_numpy(float),
    "Fusion": test_eval["fusion_probability"].to_numpy(float)
}
test_metrics = pd.DataFrame([
    {"model": name, "operating_point":"fixed_0.50",
     **classification_metrics(y_test, prob, THRESHOLD_FIXED)}
    for name, prob in model_probs.items()
] + [
    {"model": name, "operating_point":"development_LOO_Youden",
     **classification_metrics(y_test, prob, THRESHOLD_YOUDEN_DEV)}
    for name, prob in model_probs.items()
])
display(test_metrics.round(4))
dev_loo_metrics = pd.DataFrame([
    {"operating_point":"fixed_0.50", **classification_metrics(y_dev, loo_prob, 0.5)},
    {"operating_point":"development_LOO_Youden", **classification_metrics(y_dev, loo_prob, THRESHOLD_YOUDEN_DEV)}
])
print("Development fusion-LOO metrics (threshold selection/evaluation only):")
display(dev_loo_metrics.round(4))


Matched labelled test participants: 46
Unmatched test participants: [334]
Test class counts: {0: 32, 1: 14}


,model,operating_point,threshold,AUC,PR_AUC,accuracy,sensitivity,specificity,PPV,NPV,F1,TN,FP,FN,TP,sensitivity_95CI,specificity_95CI,PPV_95CI,NPV_95CI,Brier
0,Acoustic,fixed_0.50,0.5000,0.5402,0.3787,0.5000,0.3571,0.5625,0.2632,0.6667,0.3030,18,14,9,5,"(0.1634473175355414, 0.612355768960926)","(0.3932559113685198, 0.7183466888495963)","(0.11806412346782508, 0.487915508978151)","(0.47824760507320074, 0.8135673850128435)",0.2517
1,Linguistic,fixed_0.50,0.5000,0.8036,0.5735,0.6957,0.0714,0.9688,0.5000,0.7045,0.1250,31,1,13,1,"(0.012722215247890939, 0.31468704424151117)","(0.8425573617998744, 0.9944621398359969)","(0.09453120573423074, 0.9054687942657693)","(0.5577979987383977, 0.8184447088149512)",0.1770
2,Fusion,fixed_0.50,0.5000,0.7210,0.5456,0.6957,0.5714,0.7500,0.5000,0.8000,0.5333,24,8,6,8,"(0.3259064457965074, 0.7861920109552588)","(0.5789344100575552, 0.8674759908149096)","(0.27999563610326017, 0.7200043638967398)","(0.6269430358685175, 0.9049489282271013)",0.2258
3,Acoustic,development_LOO_Youden,0.5801,0.5402,0.3787,0.6087,0.1429,0.8125,0.2500,0.6842,0.1818,26,6,12,2,"(0.040093921271464966, 0.39941379496970353)","(0.6469084479862404, 0.9111045531043405)","(0.071479212752109, 0.5907245696898311)","(0.5254424010939046, 0.8091539632440382)",0.2517
4,Linguistic,development_LOO_Youden,0.5801,0.8036,0.5735,0.6957,0.0000,1.0000,NaN,0.6957,0.0000,32,0,14,0,"(1.3877787807814457e-17, 0.2153108027376358)","(0.8928208017449293, 1.0)","(nan, nan)","(0.5519435355159235, 0.8092015919723277)",0.1770
5,Fusion,development_LOO_Youden,0.5801,0.7210,0.5456,0.6739,0.2143,0.8750,0.4286,0.7179,0.2857,28,4,11,3,"(0.07571386712693184, 0.47589230586600295)","(0.7193169451834978, 0.9502986561251991)","(0.15821985525146975, 0.7495416354723428)","(0.5622468051697918, 0.8345650791115233)",0.2258


Development fusion-LOO metrics (threshold selection/evaluation only):


,operating_point,threshold,AUC,PR_AUC,accuracy,sensitivity,specificity,PPV,NPV,F1,TN,FP,FN,TP,sensitivity_95CI,specificity_95CI,PPV_95CI,NPV_95CI,Brier
0,fixed_0.50,0.5000,0.6632,0.5719,0.6729,0.5667,0.7143,0.4359,0.8088,0.4928,55,22,13,17,"(0.3919730700081361, 0.7262251442353347)","(0.6051015707559996, 0.8031048184086717)","(0.2930475387568963, 0.5902430835133052)","(0.699893585399794, 0.8847270596907507)",0.2216
1,development_LOO_Youden,0.5801,0.6632,0.5719,0.7757,0.5000,0.8831,0.6250,0.8193,0.5556,68,9,15,15,"(0.33154125640533766, 0.6684587435946623)","(0.7925491643206027, 0.937274379943506)","(0.42709962442674276, 0.7884063244045122)","(0.7230012316563488, 0.8873063417048004)",0.2216


## 5. Uncertainty and paired comparisons

AUC confidence intervals and paired AUC differences use stratified participant-level bootstrap resampling. McNemar's exact test compares paired correctness at the same fixed 0.5 threshold.

In [7]:
rng = np.random.default_rng(SEED+2026)
pos = np.where(y_test == 1)[0]
neg = np.where(y_test == 0)[0]
if len(pos) == 0 or len(neg) == 0:
    raise ValueError("Held-out matched test cohort must contain both PHQ_Binary classes for AUC and stratified bootstrap.")
boot = {k: [] for k in model_probs}
boot_delta = {"Fusion_vs_Linguistic": [], "Fusion_vs_Acoustic": []}
for _ in range(N_BOOT):
    ix = np.concatenate([
        rng.choice(pos, len(pos), replace=True),
        rng.choice(neg, len(neg), replace=True)
    ])
    aucs = {k: roc_auc_score(y_test[ix], p[ix]) for k,p in model_probs.items()}
    for k in model_probs: boot[k].append(aucs[k])
    boot_delta["Fusion_vs_Linguistic"].append(aucs["Fusion"]-aucs["Linguistic"])
    boot_delta["Fusion_vs_Acoustic"].append(aucs["Fusion"]-aucs["Acoustic"])

ci_rows = []
for k, vals in boot.items():
    lo, hi = np.quantile(vals, [0.025,0.975])
    ci_rows.append({"model": k, "AUC_CI_low": lo, "AUC_CI_high": hi})
auc_ci = pd.DataFrame(ci_rows)
paired_auc = pd.DataFrame([
    {"comparison": k, "delta_AUC": float(np.mean(v)),
     "paired_bootstrap_CI_low": float(np.quantile(v,0.025)),
     "paired_bootstrap_CI_high": float(np.quantile(v,0.975))}
    for k,v in boot_delta.items()
])

from scipy.stats import binomtest
def mcnemar_exact(p1, p2, threshold=THRESHOLD_FIXED):
    c1 = (p1 >= threshold) == y_test
    c2 = (p2 >= threshold) == y_test
    b = int(np.sum(c1 & ~c2))
    c = int(np.sum(~c1 & c2))
    return {"fusion_only_correct": b, "comparator_only_correct": c,
            "McNemar_exact_p": float(binomtest(min(b,c), b+c, 0.5).pvalue) if b+c else 1.0}

paired_class = pd.DataFrame([
    {"comparison":"Fusion vs Linguistic @0.50", **mcnemar_exact(model_probs["Fusion"],model_probs["Linguistic"],THRESHOLD_FIXED)},
    {"comparison":"Fusion vs Acoustic @0.50", **mcnemar_exact(model_probs["Fusion"],model_probs["Acoustic"],THRESHOLD_FIXED)},
    {"comparison":"Fusion vs Linguistic @development Youden", **mcnemar_exact(model_probs["Fusion"],model_probs["Linguistic"],THRESHOLD_YOUDEN_DEV)},
    {"comparison":"Fusion vs Acoustic @development Youden", **mcnemar_exact(model_probs["Fusion"],model_probs["Acoustic"],THRESHOLD_YOUDEN_DEV)}
])
display(auc_ci.round(4)); display(paired_auc.round(4)); display(paired_class.round(4))

,model,AUC_CI_low,AUC_CI_high
0,Acoustic,0.3616,0.7054
1,Linguistic,0.6629,0.9263
2,Fusion,0.5536,0.8684


,comparison,delta_AUC,paired_bootstrap_CI_low,paired_bootstrap_CI_high
0,Fusion_vs_Linguistic,-0.0840,-0.2076,0.0246
1,Fusion_vs_Acoustic,0.1822,0.0357,0.3482


,comparison,fusion_only_correct,comparator_only_correct,McNemar_exact_p
0,Fusion vs Linguistic @0.50,7,7,1.0000
1,Fusion vs Acoustic @0.50,14,5,0.0636
2,Fusion vs Linguistic @development Youden,3,4,1.0000
3,Fusion vs Acoustic @development Youden,6,3,0.5078


## 6. Calibration diagnostics and PHQ-8 severity

Calibration is assessed only; no post-hoc calibration transform is fitted to the test set. Severity predictions come from the existing multi-task regression head, with sigmoid output scaled to 0–24. Isotonic regression is not used.

In [8]:
def calibration_summary(y, p, n_bins=5):
    p = np.clip(np.asarray(p,float),1e-6,1-1e-6)
    z = np.log(p/(1-p)).reshape(-1,1)
    if len(np.unique(y)) < 2:
        intercept, slope = np.nan, np.nan
    else:
        lr = LogisticRegression(C=1e6,max_iter=2000).fit(z,y)
        intercept, slope = float(lr.intercept_[0]), float(lr.coef_[0,0])
    n_unique = len(np.unique(p))
    if n_unique < 2:
        bins = pd.Series(["all"] * len(p))
    else:
        bins = pd.qcut(p, q=min(n_bins,n_unique), duplicates="drop")
    tab = pd.DataFrame({"p":p,"y":y,"bin":bins}).groupby("bin",observed=True).agg(
        mean_predicted=("p","mean"), observed_fraction=("y","mean"), n=("y","size")
    ).reset_index(drop=True)
    ece = float(np.sum(tab.n/len(y)*np.abs(tab.mean_predicted-tab.observed_fraction)))
    return {"Brier":brier_score_loss(y,p),
            "calibration_intercept":intercept,
            "calibration_slope":slope,"ECE":ece}, tab

cal_rows = []
for name,p in model_probs.items():
    sm, tab = calibration_summary(y_test,p)
    cal_rows.append({"model":name,**sm})
calibration_metrics = pd.DataFrame(cal_rows)
display(calibration_metrics.round(4))

sev = test_eval["fusion_severity_PHQ8"].to_numpy(float)
severity_metrics = pd.DataFrame([{
    "MAE":float(np.mean(np.abs(sev-score_test))),
    "RMSE":float(np.sqrt(np.mean((sev-score_test)**2))),
    "R2":float(1-np.sum((sev-score_test)**2)/np.sum((score_test-score_test.mean())**2)) if np.sum((score_test-score_test.mean())**2) > 0 else np.nan
}])
display(severity_metrics.round(4))

,model,Brier,calibration_intercept,calibration_slope,ECE
0,Acoustic,0.2517,-0.8087,0.2949,0.2488
1,Linguistic,0.1770,0.9745,2.3306,0.1213
2,Fusion,0.2258,-0.8098,1.8904,0.1834


,MAE,RMSE,R2
0,6.3682,7.1078,-0.2085


## 7. Ablations using the same frozen branch inputs

These are fusion-head ablations only; the acoustic and linguistic branches remain unchanged. They are evaluated on the same independent test participants and fixed threshold. They must be described as such, not as branch ablations.

In [9]:
# Two-feature fusion: p_acoustic and p_linguistic only.
X2_dev = make_meta_features(pa_dev,pl_dev,"two")
X2_test = make_meta_features(pa_test,pl_test,"two")
two_prob, two_sev, two_models = fit_ensemble(X2_dev,y_dev,s_dev,X2_test,alpha=ALPHA)

# No-severity-loss ablation: same four inputs and MLP, classification-only objective.
no_sev_prob, no_sev_pred, no_sev_models = fit_ensemble(X_dev,y_dev,s_dev,X_test,alpha=0.0)

ablation_metrics = pd.DataFrame([
    {"model":"Full fusion", **classification_metrics(y_test,model_probs["Fusion"])},
    {"model":"Two-feature fusion", **classification_metrics(y_test,two_prob)},
    {"model":"Fusion without severity loss", **classification_metrics(y_test,no_sev_prob)}
])
display(ablation_metrics.round(4))

,model,threshold,AUC,PR_AUC,accuracy,sensitivity,specificity,PPV,NPV,F1,TN,FP,FN,TP,sensitivity_95CI,specificity_95CI,PPV_95CI,NPV_95CI,Brier
0,Full fusion,0.5,0.7210,0.5456,0.6957,0.5714,0.7500,0.5000,0.8000,0.5333,24,8,6,8,"(0.3259064457965074, 0.7861920109552588)","(0.5789344100575552, 0.8674759908149096)","(0.27999563610326017, 0.7200043638967398)","(0.6269430358685175, 0.9049489282271013)",0.2258
1,Two-feature fusion,0.5,0.6920,0.4871,0.6522,0.5714,0.6875,0.4444,0.7857,0.5000,22,10,6,8,"(0.3259064457965074, 0.7861920109552588)","(0.5143332169904459, 0.8204745836639026)","(0.24559518127005633, 0.6628358435795723)","(0.6046141434627812, 0.8978753966710211)",0.2340
2,Fusion without severity loss,0.5,0.7188,0.5445,0.6957,0.5714,0.7500,0.5000,0.8000,0.5333,24,8,6,8,"(0.3259064457965074, 0.7861920109552588)","(0.5789344100575552, 0.8674759908149096)","(0.27999563610326017, 0.7200043638967398)","(0.6269430358685175, 0.9049489282271013)",0.2264


## 8. Export tables, predictions, and reproducibility manifest

Complete the TRIPOD+AI checklist against the revised manuscript. A generated checklist is not evidence that the manuscript already complies.

In [10]:
test_metrics.to_csv(os.path.join(WORKING,"fusion_frozen_test_metrics.csv"),index=False)
auc_ci.to_csv(os.path.join(WORKING,"fusion_frozen_auc_confidence_intervals.csv"),index=False)
paired_auc.to_csv(os.path.join(WORKING,"fusion_frozen_paired_auc_bootstrap.csv"),index=False)
paired_class.to_csv(os.path.join(WORKING,"fusion_frozen_mcnemar.csv"),index=False)
calibration_metrics.to_csv(os.path.join(WORKING,"fusion_frozen_calibration_diagnostics.csv"),index=False)
severity_metrics.to_csv(os.path.join(WORKING,"fusion_frozen_severity_metrics.csv"),index=False)
ablation_metrics.to_csv(os.path.join(WORKING,"fusion_frozen_ablation_metrics.csv"),index=False)
test_eval.to_csv(os.path.join(WORKING,"fusion_frozen_test_predictions_with_labels.csv"),index=False)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

manifest = {
    "evaluation_protocol":"OOF-stacked fusion trained on 107 development participants; one independent held-out test evaluation",
    "nested_full_pipeline_LOO_claim":False,
    "branch_models_retrained":False,
    "development_n":int(len(dev)),
    "matched_test_n":int(len(test_eval)),
    "unmatched_test_ids":sorted(set(lab.Participant_ID)-set(test_eval.Participant_ID)),
    "fusion_architecture":"4 -> 16 -> 8 MLP; BatchNorm/ReLU/Dropout 0.3; binary and PHQ-8 heads",
    "meta_features":["acoustic_probability","linguistic_probability","absolute_difference","product"],
    "loss":"weighted BCE + 0.3 * MSE(sigmoid(severity), PHQ8/24)",
    "optimizer":"AdamW", "learning_rate":LR, "weight_decay":WEIGHT_DECAY,
    "epochs":EPOCHS, "seeds":N_SEEDS, "thresholds":{"fixed":0.5,"development_LOO_Youden":float(THRESHOLD_YOUDEN_DEV)},
    "calibration":"diagnostic only; no test-fitted calibration",
    "severity":"sigmoid output scaled to 0..24; no isotonic regression",
    "inputs_sha256":{os.path.basename(p):sha256(p) for p in [p_aoof,p_loof,p_atest] + ([p_ltest] if p_ltest else [p_lbundle]) + [p_labels]},
    "software":{"python":platform.python_version(),"torch":torch.__version__,
                "numpy":np.__version__,"pandas":pd.__version__}
}
with open(os.path.join(WORKING,"fusion_frozen_run_manifest.json"),"w") as f:
    json.dump(manifest,f,indent=2)

tripod_items = [
"Title and abstract identify model development/evaluation",
"Data source, setting, dates and eligibility described",
"Outcome definition and timing described",
"Predictors and measurement procedures specified",
"Sample size and participant flow reported",
"Missing audio/transcript handling and participant 334 exclusion reported",
"Preprocessing and feature engineering specified",
"Model architecture, hyperparameters and software versions reported",
"OOF branch predictions and stacking protocol described accurately",
"Limits of non-nested branch/fusion validation disclosed",
"Threshold fixed before test evaluation; no test tuning",
"Discrimination, calibration diagnostics and operating metrics reported with uncertainty",
"Internal validation and held-out test evaluation clearly distinguished",
"Subgroup/fairness limitations discussed where data permit",
"Code, implementation details and data-access constraints documented"
]
pd.DataFrame({"TRIPOD+AI reporting item":tripod_items,
              "manuscript_status":["AUTHOR TO VERIFY"]*len(tripod_items)}).to_csv(
    os.path.join(WORKING,"TRIPOD_AI_author_completion.csv"),index=False)

print("Exported metrics, predictions, manifest, and TRIPOD+AI author checklist to:",WORKING)

Exported metrics, predictions, manifest, and TRIPOD+AI author checklist to: /kaggle/working


## 9. Manuscript interpretation guardrails

1. State that the acoustic and linguistic branch models were frozen for this fusion analysis.
2. Describe the 107 branch OOF predictions as stacking inputs, not as nested outer-fold predictions.
3. Report held-out test metrics on the **46 matched participants**; participant 334 has no matched audio/transcript prediction and must be accounted for in the participant flow.
4. Do not claim fusion outperforms the linguistic branch unless the paired held-out comparison supports that claim. AUC point estimates and confidence intervals must be shown.
5. Describe PHQ-8 regression as exploratory severity estimation, not a clinical severity diagnosis.
6. Do not call probabilities calibrated merely because a calibration diagnostic was computed.
7. The full nested LOO requirement remains unmet with frozen branches. If the editor/reviewer insists on complete nested LOO for every branch and fusion component, this frozen-branch protocol cannot satisfy that specific request without new fold-wise branch fitting.